# 08 · Response time on the 40 questions (US-11, CHUNKY-133)

Added by barissaka for CHUNKY-191 (EN-33 Measure the response time on the 40 questions and optimise if needed), a task of US-11 (CHUNKY-133 Read the full answer within one minute).

Acceptance criterion measured here: when the user submits a question, the assistant shall display the complete answer within 60 seconds.

The notebook asks each of the 40 questions of `docs/eval/Eval_Set_1.csv` once, with the settings of `config/rag.yaml`, and records the time from the question to the complete answer (`seconds` of `chunky.answer.answer`: retrieval on this laptop plus the model call). The interface adds no model call of its own, so this is the time the user waits.

The run includes the test questions. Only the times are used; no prompt or setting is changed from what is seen here.

LLM calls: 40. The run takes about 5 minutes.

Runs:
- 24 Nov 2026, first run (`docs/eval/runs/2026-11-24_response_time.jsonl`): 38 of 40 within 60 seconds, median 7.1 s. S01 (83.3 s) and S25 (75.1 s) were over the limit. The same questions took 6-11 s in other runs, and their answers were not longer than the others: the model service sometimes stalls for about a minute.
- After that, `call_llm` in `src/chunky/answer.py` got a time limit of 25 s and one retry. Runs made with it are saved with `RUN_LABEL` in the file name, so the first run is kept for comparison.

In [ ]:
import csv
import json
import re
import statistics
import time
from datetime import date

from chunky.answer import answer
from chunky.index import ROOT, Embedder, load_config, load_index

LIMIT_S = 60   # acceptance criterion of US-11: complete answer within 60 seconds
RUN_LABEL = "with_retry"   # added to the name of the result file; change it for a new run to keep the earlier files

cfg = load_config()
emb = cfg["embedding"]
embedder = Embedder(emb["model"], emb["revision"], emb["max_tokens"])
index = load_index()

# Some questions end with their source, e.g. "(EGM SSS-4)"; a user would not type it, so it is removed.
SOURCE_NOTE = re.compile(r"\s*\(EGM SSS-\d+\)\s*$")
with open(ROOT / "docs/eval/Eval_Set_1.csv", encoding="utf-8-sig", newline="") as f:
    questions = [(q["Question ID"], SOURCE_NOTE.sub("", q["Question"])) for q in csv.DictReader(f)]
gen, ret = cfg["generation"], cfg["retrieval"]
print(len(questions), "questions | model:", gen["model"], "| prompt:", gen["prompt"],
      "| top_k:", ret["top_k"], "| rerank:", ret.get("rerank"))

In [ ]:
out = ROOT / "docs/eval/runs" / f"{date.today()}_response_time_{RUN_LABEL}.jsonl"
out.parent.mkdir(parents=True, exist_ok=True)
rows = []
with open(out, "w", encoding="utf-8") as f:
    for qid, text in questions:
        start = time.time()
        try:
            r = answer(text, embedder, index, cfg)
            r["error"] = None
        except Exception as e:  # a failed call is recorded with its waiting time and counts as over the limit
            r = {"question": text, "answer": None, "seconds": round(time.time() - start, 1),
                 "error": f"{type(e).__name__}: {e}"}
        r["question_id"] = qid
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
        rows.append(r)
        print(f"{qid}: {r['seconds']:5.1f} s" + ("  (sent again)" if (r.get("llm_attempts") or 1) > 1 else "")
              + ("  ERROR " + r["error"] if r["error"] else ""))
print("saved:", out)

In [ ]:
def summary(label, part):
    """Print the time statistics of a group of answers.

    Args:
        label (str): name of the group shown at the start of the line.
        part (list[dict]): result rows with the fields "seconds" and "error".

    Returns:
        None. Prints the number of answers, the median, the 95th percentile, the maximum
        and how many answers came within LIMIT_S seconds without an error.
    """
    times = sorted(r["seconds"] for r in part)
    p95 = times[max(0, -(-len(times) * 95 // 100) - 1)]  # nearest-rank 95th percentile
    within = sum(1 for r in part if r["seconds"] <= LIMIT_S and not r["error"])
    print(f"{label:22} n={len(times):2}  median {statistics.median(times):5.1f} s  p95 {p95:5.1f} s  "
          f"max {times[-1]:5.1f} s  within {LIMIT_S} s: {within}/{len(times)}")


summary("all 40 questions", rows)
summary("in scope (S01-S30)", [r for r in rows if r["question_id"].startswith("S")])
summary("out of scope (K01-K10)", [r for r in rows if r["question_id"].startswith("K")])

print("\nslowest five:")
for r in sorted(rows, key=lambda r: -r["seconds"])[:5]:
    print(f"  {r['question_id']}: {r['seconds']:5.1f} s  {r['question'][:70]}")

retried = [r["question_id"] for r in rows if (r.get("llm_attempts") or 1) > 1]
print(f"\nmodel call sent again after {cfg['generation'].get('timeout_s', 25)} s without a reply:",
      f"{len(retried)}/40" + (f" ({', '.join(retried)})" if retried else ""))

late = [r["question_id"] for r in rows if r["seconds"] > LIMIT_S or r["error"]]
print(f"\nUS-11 criterion (complete answer within {LIMIT_S} s):",
      "met for 40/40" if not late else f"NOT met for {len(late)}/40: {', '.join(late)}")